In [2]:
##############################################################################
# Cell 1: Configuration & Setup
#
# HYBRID APPROACH:
#   - Selenium for search (handles CAPTCHA/JS challenge in visible browser)
#   - requests for detail pages (fast, no browser needed, bulk of the work)
#
# HOW TO GET COOKIES (for requests session):
#   1. Open https://intezmenykereso.mnb.hu/en/Home/Index in Chrome
#   2. Open DevTools (F12) -> Application tab -> Cookies
#   3. Copy the 3 cookie values below
#   4. Cookies are session-scoped; refresh them if the script reports expiry
##############################################################################

import pandas as pd
import requests
from time import sleep
import datetime
import os
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'HU CBH'
print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
#writer = ExcelWriter(filename)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

print("Running HU CBH Web Scraping Tool v2.0 (hybrid: Selenium search + requests details)")

# =============================================================================
# PASTE YOUR COOKIES HERE (from browser DevTools -> Application -> Cookies)
# These are used by the requests session for fast detail-page fetching.
# =============================================================================
COOKIES = {
    "__RequestVerificationToken": "rov5veA_jQtlW4-Sw7koRlb8S-t6jD9WWPSe02XZuAv-zxLbjqWT9mapTx0N_AFsNTH4dqvonp2SxjRiWQn9h-G6Cd6vcpcxJGS7L15NVw01",
    "LBSESSSION": "!kRaH4xgVpZBIXz+SPPl0w8iG54e73HCHNoc9XZg9m1eU0qE+Ov2Pbu3y//8m7j82q1rrxlhHL4P9AA==",
    "TS0100c62d": "012f7c1fffb87e9318eeeccdfd63b1394bf8f0f943ad34bbf1b2d864c476ff76022838d486a09dcd3ddf87dc33a4475ffde292db042e0488529390108b366e000adf788af6592894dcd60169aaabd08dd34bb236f302bf2adf0f20b42e014ce98e295bff0b",
}
# =============================================================================

regulatorName = 'HU CBH'
scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
os.chdir(scriptfolder)

now = datetime.datetime.now()
filename = 'HU CBH data {}.xlsx'.format(str(now).replace(":", ".")[:-7])

BASE_URL = "https://intezmenykereso.mnb.hu"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "Connection": "keep-alive",
    "sec-ch-ua": '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "sec-fetch-dest": "document",
    "sec-fetch-mode": "navigate",
    "sec-fetch-site": "same-origin",
    "upgrade-insecure-requests": "1",
}

# Set up requests session with cookies for fast detail-page fetching
req_session = requests.Session()
req_session.headers.update(HEADERS)
for name, value in COOKIES.items():
    req_session.cookies.set(name, value, domain="intezmenykereso.mnb.hu")

# Validate cookies
resp = req_session.get(f"{BASE_URL}/en/Home/Index", timeout=15)
if resp.status_code == 200 and "complex-search-panel" in resp.text:
    print("Requests session valid - cookies accepted.")
    USE_REQUESTS_FOR_DETAILS = True
else:
    print(f"WARNING: Cookies may be expired (status={resp.status_code}).")
    print("Detail pages will fall back to Selenium (slower).")
    USE_REQUESTS_FOR_DETAILS = False

# Regdict: maps reg codes to XPath selectors for category/subcategory tree nodes
# Excluded: HU CBH 4 (empty xpaths), HU CBH 18 (no subcategory)
regdict = {
    # 'HU CBH 1':  ['//*[@id="-6"]/div[2]', '//*[@id="-20"]/div[2]'],
    # 'HU CBH 2':  ['//*[@id="-6"]/div[2]', '//*[@id="-19"]/div[2]'],
    # 'HU CBH 3':  ['//*[@id="-8"]/div[2]', '//*[@id="-24"]/div[2]'],
    # 'HU CBH 5':  ['//*[@id="-8"]/div[2]', '//*[@id="132"]/div[2]'],
    # 'HU CBH 6':  ['//*[@id="-8"]/div[2]', '//*[@id="-23"]/div[2]'],
    #  'HU CBH 7':  ['//*[@id="-1"]/div[2]', '//*[@id="-10"]/div[2]'],
    #  'HU CBH 8':  ['//*[@id="-1"]/div[2]', '//*[@id="-9"]/div[2]'],
    #  'HU CBH 9':  ['//*[@id="-7"]/div[2]', '//*[@id="39"]/div[2]'],
    #  'HU CBH 10': ['//*[@id="-7"]/div[2]', '//*[@id="-22"]/div[2]'],

    #  'HU CBH 12': ['//*[@id="-5"]/div[2]', '//*[@id="-18"]/div[2]'],
    #  'HU CBH 13': ['//*[@id="-3"]/div[2]', '//*[@id="-14"]/div[2]'],
    #  'HU CBH 14': ['//*[@id="-3"]/div[2]', '//*[@id="-16"]/div[2]'],
    #  'HU CBH 15': ['//*[@id="-3"]/div[2]', '//*[@id="-15"]/div[2]'],
      'HU CBH 16': ['//*[@id="-3"]/div[2]', '//*[@id="-12"]/div[2]'],
    # 'HU CBH 17': ['//*[@id="-4"]/div[2]', '//*[@id="-17"]/div[2]'],
}

catlist = [
    'Name', 'Previous name', 'Administrative address', 'Type of institution',
    'Registration number', 'Registry court/court number', 'Website address',
    'Location of publication', 'Legal status',
    '[EN]Közérdeklődésre számon tartott hitelintézet'
]

print(f"Output file: {filename}")
print(f"Reg codes to process: {len(regdict)}")





    
    

Running HU CBH Web Scraping Tool v.1.1
Running HU CBH Web Scraping Tool v2.0 (hybrid: Selenium search + requests details)
Requests session valid - cookies accepted.
Output file: HU CBH data 2026-04-03 18.58.09.xlsx
Reg codes to process: 1


In [2]:
##############################################################################
# Cell 2: Selenium search function
# Uses browser ONLY for the search step (handles JS challenge / overlays).
# Extracts LId values and Authorization from search results.
##############################################################################

def selenium_search(driver, reg, xpaths, wait):
    """
    Perform a search for one reg code using Selenium.
    Returns (lids, auths) - lists of institution LId values and authorization statuses.
    """
    cat_xpath, subcat_xpath = xpaths

    driver.get(f'{BASE_URL}/en/Home/Index')
    sleep(5)

    # Use JS clicks throughout to bypass any overlay (cookie banner, CAPTCHA shield)
    # 1. Click "Advanced search" expander
    adv_search = wait.until(EC.presence_of_element_located(
        (By.XPATH, '/html/body/div[3]/div[6]')))
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", adv_search)
    driver.execute_script("arguments[0].click();", adv_search)
    sleep(1)

    # 2. Open "Institution field of activities" panel
    complex_panel = wait.until(EC.presence_of_element_located(
        (By.XPATH, '//*[@id="complex-search-panel"]/div[1]')))
    driver.execute_script("arguments[0].click();", complex_panel)
    sleep(2)

    # 3. Click category node
    cat_el = wait.until(EC.presence_of_element_located((By.XPATH, cat_xpath)))
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", cat_el)
    driver.execute_script("arguments[0].click();", cat_el)
    sleep(1)

    # 4. Click subcategory node
    subcat_el = wait.until(EC.presence_of_element_located((By.XPATH, subcat_xpath)))
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", subcat_el)
    driver.execute_script("arguments[0].click();", subcat_el)
    sleep(1)

    # 5. Click search button
    search_btn = wait.until(EC.presence_of_element_located(
        (By.XPATH, '//*[@id="complex-inst-search-button"]')))
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", search_btn)
    driver.execute_script("arguments[0].click();", search_btn)

    # 6. Wait for results -- if CAPTCHA appears, solve it manually in the browser
    print(f"  Waiting for results (solve CAPTCHA in browser if prompted)...")
    captcha_timeout = 300
    poll_interval = 3
    elapsed = 0
    count_el = None
    total_el = None
    while elapsed < captcha_timeout:
        sleep(poll_interval)
        elapsed += poll_interval
        source = driver.page_source
        tempstr = BeautifulSoup(source, 'html.parser')
        count_el = tempstr.find('text', {'id': 'count'})
        total_el = tempstr.find('text', {'id': 'total-count'})
        if count_el and total_el:
            break
        if elapsed % 15 == 0:
            print(f"    Still waiting... ({elapsed}s elapsed, solve CAPTCHA in browser)")

    if not count_el or not total_el:
        print(f"  WARNING: No results after {captcha_timeout}s for {reg}. Skipping.")
        return [], []

    count = int(count_el.text)
    tcount = int(total_el.text)
    print(f"  Found {count}/{tcount} results loaded...")

    max_retries = 10
    retry = 0
    while count < tcount and retry < max_retries:
        retry += 1
        driver.execute_script(
            "var s = (document.scrollingElement || document.body);"
            "s.scrollTop = s.scrollHeight;")
        sleep(1)

        # Try multiple strategies to find and click "Load More"
        clicked = False
        # Strategy 1: find any input[type=button] with "Load" in its value
        try:
            btns = driver.find_elements(By.CSS_SELECTOR, '#result-table input[type="button"]')
            for btn in btns:
                val = btn.get_attribute("value") or ""
                if "load" in val.lower() or "more" in val.lower() or "további" in val.lower():
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
                    driver.execute_script("arguments[0].click();", btn)
                    clicked = True
                    break
        except Exception:
            pass

        # Strategy 2: original XPath with JS click
        if not clicked:
            try:
                btn = driver.find_element(By.XPATH, '//*[@id="result-table"]/div[5]/div[1]/input')
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
                driver.execute_script("arguments[0].click();", btn)
                clicked = True
            except Exception:
                pass

        # Strategy 3: click via pure JS DOM query
        if not clicked:
            try:
                driver.execute_script("""
                    var inputs = document.querySelectorAll('#result-table input[type="button"]');
                    for (var i = 0; i < inputs.length; i++) {
                        inputs[i].click();
                    }
                """)
                clicked = True
            except Exception:
                pass

        sleep(2)

        # Re-read count
        source = driver.page_source
        tempstr = BeautifulSoup(source, 'html.parser')
        new_count_el = tempstr.find('text', {'id': 'count'})
        if new_count_el:
            new_count = int(new_count_el.text)
            if new_count > count:
                print(f"  Loaded {new_count}/{tcount}...")
                count = new_count
                retry = 0
            elif not clicked:
                print(f"  Retry {retry}: button not found, count still {count}/{tcount}")

    if count < tcount:
        print(f"  WARNING: Only loaded {count}/{tcount} results after {max_retries} retries.")

    # 7. Parse all result rows
    source = driver.page_source
    tempstr = BeautifulSoup(source, 'html.parser')
    rows = tempstr.find_all('div', {'class': 'result-table-row'})

    lids = []
    auths = []
    for row in rows:
        # Find authorization by looking for the auth-related button in the row
        auth_input = (
            row.find('input', class_='activity-has-no-right-button')
            or row.find('input', class_='activity-has-right-button')
            or row.find('input', attrs={'rel': 'detailstooltip'})
        )
        if auth_input:
            auth = auth_input.get('value', '').strip()
        else:
            auth = ''
        if auth and 'Not' not in auth:
            auth = auth.split(' ')[0].strip()
        auths.append(auth)

        # Find LId by looking for the input with a lid attribute
        lid_input = row.find('input', attrs={'lid': True})
        if lid_input:
            lids.append(lid_input['lid'])
        else:
            lids.append('')

    print(f"  Extracted {len(lids)} institutions for {reg}.")
    return lids, auths

print("Search function defined.")

Search function defined.


In [3]:
##############################################################################
# Cell 3: Detail page fetcher
# Uses requests first (fast), falls back to Selenium for JS-rendered pages.
##############################################################################

def parse_detail_html(soup):
    """Parse detail page HTML. Tries standard layout, then mobile table layout."""
    tempdict = {}

    # Strategy 1: standard layout with div#details-list
    datadiv = soup.find('div', {"id": "details-list"})
    if datadiv:
        for ro in datadiv.find_all('div', {'class': "result-table-row"}):
            cells = ro.find_all("div", {"class": "result-table-cell"})
            if len(cells) >= 2:
                tempdict[cells[0].text.strip()] = cells[1].text.strip()
        if tempdict:
            return tempdict

    # Strategy 2: mobile/alternate table layout (e.g. insurance intermediaries)
    for tr in soup.find_all('tr', {'class': 'result-view-row-mobile'}):
        head = tr.find('td', {'class': 'result-view-headcell-mobile'})
        val = tr.find('td', {'class': 'result-view-cell-mobile'})
        if head and val:
            key = head.text.strip()
            inp = val.find('input', value=True)
            value = inp['value'].strip() if inp else val.text.strip()
            tempdict[key] = value
    if tempdict:
        return tempdict

    # Strategy 3: any table with key-value rows
    for table in soup.find_all('table'):
        for tr in table.find_all('tr'):
            tds = tr.find_all('td')
            if len(tds) >= 2:
                tempdict[tds[0].text.strip()] = tds[1].text.strip()
    return tempdict if tempdict else None


def fetch_detail_requests(lid, req_session):
    """Fetch institution detail page using requests. Returns parsed dict or None."""
    url = (f"{BASE_URL}/en/Details/Index?LId={lid}"
           f"&EntityType=Institute&expandAccordions=IntezmenyAlapadatok")
    try:
        resp = req_session.get(url, timeout=15)
        if resp.status_code != 200:
            return None
        soup = BeautifulSoup(resp.text, 'html.parser')
        return parse_detail_html(soup)
    except Exception as e:
        print(f"    requests error for LId={lid}: {e}")
        return None


def fetch_detail_selenium(driver, lid):
    """Fetch institution detail page using Selenium. Waits for JS to render."""
    url = (f"{BASE_URL}/en/Details/Index?LId={lid}"
           f"&EntityType=Institute&expandAccordions=IntezmenyAlapadatok")
    driver.get(url)
    for _ in range(10):
        sleep(1)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        result = parse_detail_html(soup)
        if result:
            return result
    return None


def fetch_detail(lid, req_session, driver):
    """Fetch detail page: try requests first, fall back to Selenium."""
    if USE_REQUESTS_FOR_DETAILS:
        result = fetch_detail_requests(lid, req_session)
        if result is not None:
            return result
    return fetch_detail_selenium(driver, lid)


print("Detail fetch functions defined.")

Detail fetch functions defined.


In [4]:
##############################################################################
# Cell 4: Main loop - Search + Fetch details for all reg codes
##############################################################################

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 15)

# Also transfer cookies from requests session to Selenium browser
driver.get(f'{BASE_URL}/en/Home/Index')
sleep(3)
for name, value in COOKIES.items():
    if value and value != "https://cb.gov.sy/index.php?page=show&ex=2&dir=items&lang=2&ser=1&cat_id=1372&act=1372&":
        driver.add_cookie({
            'name': name,
            'value': value,
            'domain': 'intezmenykereso.mnb.hu',
            'path': '/'
        })

xname, xpname, xadmadd, xtypeinst = [], [], [], []
xregnum, xregcourt, xweb, xlocpub = [], [], [], []
xlegsta, xen, xauth, reglist = [], [], [], []

for reg_idx, (reg, xpaths) in enumerate(regdict.items()):
    print(f"\n[{reg_idx+1}/{len(regdict)}] Working with {reg}...")

    lids, auths = selenium_search(driver, reg, xpaths, wait)

    if not lids:
        print(f"  No results for {reg}, skipping.")
        continue

    total_firms = len(lids)
    for i, lid in enumerate(lids):
        print(f"  Fetching firm {i+1}/{total_firms} (LId={lid})...", end=' ')

        tempdict = fetch_detail(lid, req_session, driver)

        if tempdict is None:
            print("datadiv is None.")
            tempdict = {}

        for ca in catlist:
            if ca not in tempdict:
                tempdict[ca] = ''

        xname.append(tempdict[catlist[0]])
        xpname.append(tempdict[catlist[1]])
        xadmadd.append(tempdict[catlist[2]])
        xtypeinst.append(tempdict[catlist[3]])
        xregnum.append(tempdict[catlist[4]])
        xregcourt.append(tempdict[catlist[5]])
        xweb.append(tempdict[catlist[6]])
        xlocpub.append(tempdict[catlist[7]])
        xlegsta.append(tempdict[catlist[8]])
        xen.append(tempdict[catlist[9]])
        xauth.append(auths[i])
        reglist.append(reg)
        print("OK")

print(f"\nTotal firms collected: {len(xname)}")
print("Search and detail fetching complete.")


[1/1] Working with HU CBH 16...
  Waiting for results (solve CAPTCHA in browser if prompted)...
  Found 10/3949 results loaded...
  Loaded 35/3949...
  Loaded 60/3949...
  Loaded 85/3949...
  Loaded 110/3949...
  Loaded 135/3949...
  Loaded 160/3949...
  Loaded 185/3949...
  Loaded 210/3949...
  Loaded 235/3949...
  Loaded 260/3949...
  Loaded 285/3949...
  Loaded 310/3949...
  Loaded 335/3949...
  Loaded 360/3949...
  Loaded 385/3949...
  Loaded 410/3949...
  Loaded 435/3949...
  Loaded 460/3949...
  Loaded 485/3949...
  Loaded 510/3949...
  Loaded 535/3949...
  Loaded 560/3949...
  Loaded 585/3949...
  Loaded 610/3949...
  Loaded 635/3949...
  Loaded 660/3949...
  Loaded 685/3949...
  Loaded 710/3949...
  Loaded 735/3949...
  Loaded 760/3949...
  Loaded 785/3949...
  Loaded 810/3949...
  Loaded 835/3949...
  Loaded 860/3949...
  Loaded 885/3949...
  Loaded 910/3949...
  Loaded 935/3949...
  Loaded 960/3949...
  Loaded 985/3949...
  Loaded 1010/3949...
  Loaded 1035/3949...
  Loaded 

In [5]:
##############################################################################
# Cell 5: Build DataFrame and export to Excel
##############################################################################

df = pd.DataFrame({
    'Reg List': reglist,
    'Name': xname,
    'Previous name': xpname,
    'Administrative address': xadmadd,
    'Authorization': xauth,
    'Type of institution': xtypeinst,
    'Registration number': xregnum,
    'Registry court/court number': xregcourt,
    'Website address': xweb,
    'Location of publication': xlocpub,
    'Legal status': xlegsta,
    '[EN]Közérdeklődésre számon tartott hitelintézet': xen
})

# df.to_excel(filename, index=False)
print(f"Data exported to: {filename}")
print(f"Total rows: {len(df)}")

driver.quit()
print("Browser closed. Done.")

Data exported to: HU CBH data 2026-04-02 10.08.42.xlsx
Total rows: 3949
Browser closed. Done.


In [33]:
##############################################################################
# Cell 6: Build df_total in standard SQL export format
##############################################################################

processdate = now.strftime('%Y-%m-%d')

Typology = {
    'HU CBH 1':  'Financial Institutions',
    'HU CBH 2':  'Non-Financial Institutions',
    'HU CBH 3':  'Investment Firm',
    'HU CBH 5':  'Trust',
    'HU CBH 6':  'Fund Manager',
    'HU CBH 7':  'Insurance Institution',
    'HU CBH 8':  'Insurance Intermediaries',
    'HU CBH 9':  'Private pension fund',
    'HU CBH 10': 'Voluntary Funds',
    #'HU CBH 11': 'List of "Foreign institutions providing cross-border services"',
    'HU CBH 12': 'Branches of Hungarian Institutions : Money Market Sector',
    'HU CBH 13': 'Foreign Institutions providing cross-border services : Money Market Sector',
    'HU CBH 14': 'Foreign Institutions providing cross-border services : Capital Market Sector',
    'HU CBH 15': 'Foreign Institutions providing cross-border services : Fund Sector',
    'HU CBH 16': 'Foreign Institutions providing cross-border services : Insurance Sector',
    'HU CBH 17': 'Representative Offices : Money Market Sector',
}

def bourange_same_length_array(sqldict):
    maxlen = max(len(v) for v in sqldict.values()) if sqldict else 0
    for key in sqldict:
        if len(sqldict[key]) < maxlen:
            sqldict[key] += [''] * (maxlen - len(sqldict[key]))
    return sqldict

sqldict = {
    'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [],
    'Name': [], 'InternalID_1': [], 'InternalID_1_type': [],
    'InternalID_2': [], 'InternalID_2_type': [],
    'InternalID_3': [], 'InternalID_3_type': [],
    'CoType': [], 'License_Type': [],
    'Address_1': [], 'Address_2': [], 'City': [], 'Zip': [], 'Cntry': [],
    'Phone': [], 'Fax': [], 'Website': [], 'Email': [],
    'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [],
    'CancellationDate': [], 'RegCtry': [], 'RegCode': [],
    'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [],
    'ListName': [], 'ListProcessDate': [],
    'LEI Code': [], 'BIC SWIFT Code': [],
    'Name - Mother Company': [],
    'Address_1 - Mother company': [], 'Address_2 -  Mother company': [],
    'City - Mother company': [], 'Zip - Mother company': [],
    'Cntry - Mother company': [],
    'Phone - Mother company': [], 'Check': []
}

for index, row in df.iterrows():
    reg = row['Reg List']
    reg_parts = reg.split(' ')

    sqldict['Name'].append(row.get('Name', ''))
    sqldict['Name - Mother Company'].append(row.get('Previous name', '').replace('n.a', ''))
    sqldict['InternalID_1'].append(row.get('Registration number', ''))
    sqldict['InternalID_1_type'].append('Registration number' if row.get('Registration number', '') else '')
    # sqldict['InternalID_2'].append(row.get('Registry court/court number', '').replace('n.a', '').replace('NA.', '').replace('N.A.', '').replace('.', ''))
    # sqldict['InternalID_2_type'].append('Registry court/court number' if row.get('Registry court/court number', '') else '')
    sqldict['Typology'].append(row.get('Type of institution', ''))
    sqldict['Address_1'].append(row.get('Administrative address', ''))
    sqldict['Website'].append(row.get('Website address', '').replace('n.a', ''))
    auth_val = row.get('Authorization', '')
    if auth_val == 'Authorized':
        sqldict['RegulationType'].append('Regulated')
    else:
        sqldict['RegulationType'].append(auth_val)
    sqldict['RegulationTypeCode'].append(auth_val)
    sqldict['ListName'].append(Typology.get(reg, ''))
    sqldict['ListProcessDate'].append(processdate)
    sqldict['RegCtry'].append(reg_parts[0] if len(reg_parts) >= 1 else '')
    sqldict['RegCode'].append(reg_parts[1] if len(reg_parts) >= 2 else '')
    sqldict['ListCode'].append(reg_parts[2] if len(reg_parts) >= 3 else '')
    sqldict['Cntry'].append('HU')


    sqldict = bourange_same_length_array(sqldict)

df_total = pd.DataFrame(sqldict)

import re
na_pattern = re.compile(r'^[\s,]*(?:n\.?a\.?[\s,]*)+$', re.IGNORECASE)
df_total['Address_1'] = df_total['Address_1'].apply(
    lambda x: '' if isinstance(x, str) and na_pattern.match(x.strip()) else x)

total_filename = filename.replace('.xlsx', '_total.xlsx')
os.chdir(scriptfolder)
df_total.to_excel(total_filename, index=False)
print(f"df_total exported to: {total_filename}")
print(f"Total rows: {len(df_total)}")
df_total.head()

df_total exported to: HU CBH data 2026-04-02 10.08.42_total.xlsx
Total rows: 3949


,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,,Insurance intermediary with cross-border services,,"""AKTUELL"" Raiffeisen Versicherungs-Maklerdiens...",K8805549,Registration number,,,...,,,"""AKTUELL"" Raiffeisen Versicherungs-Makle rdien...",,,,,,,
1,,,,Insurance intermediary with cross-border services,,"""Astra Brokers"" Ltd",K8805194,Registration number,,,...,,,,,,,,,,
2,,,,Insurance intermediary with cross-border services,,"""AVVI"" LTD",K8804366,Registration number,,,...,,,,,,,,,,
3,,,,Insurance intermediary with cross-border services,,"""Broker Ins"" LTD",K8804919,Registration number,,,...,,,,,,,,,,
4,,,,Insurance intermediary with cross-border services,,"""Broks Innovations"" LLC",K8807292,Registration number,,,...,,,,,,,,,,


In [7]:
df_total.to_excel(total_filename, index=False)

In [9]:
df_exp_16 = pd.read_excel('HU CBH data 2026-04-01 17.59.59_total.xlsx')

In [25]:
df_exp_16.replace('n.a', '', inplace=True)
df_exp_16.replace('NA.', '', inplace=True)
df_exp_16.replace('N.A.', '', inplace=True)
df_exp_16.replace('.', '', inplace=True)
df_exp_16.replace('n.a', '', inplace=True)
df_exp_16.replace('NA.', '', inplace=True)
df_exp_16.replace('N.A.', '', inplace=True)

In [30]:
df_total['Address_1'].replace('n.a', '', inplace=True)
df_total['Address_1'].replace('NA.', '', inplace=True)
df_total['Address_1'].replace('N.A.', '', inplace=True)
df_total['Address_1'].replace('.', '', inplace=True)
df_total['Address_1'].replace('n.a', '', inplace=True)
df_total['Address_1'].replace('NA.', '', inplace=True)
df_total['Address_1'].replace('N.A.', '', inplace=True)

In [ ]:
df_exp_16.fillna('', inplace=True)
df_exp_16['InternalID_2'].replace('0', '', inplace=True)


C:\Users\wuj1\AppData\Local\Temp\3\ipykernel_20340\1008081309.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_exp_16['InternalID_2'].replace('0', '', inplace=True)


In [36]:
df_combined = pd.concat([df_total, df_exp_16], ignore_index=True)

In [37]:
df_combined.to_excel('HU CBH data 2026-04-02 14.00.59_total.xlsx', index=False)

In [4]:
import os
import re
import pandas as pd
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

os.chdir(scriptfolder)
tempfolder = os.path.join(scriptfolder, 'tempfolder')

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

df_pdf = pd.read_excel('HU CBH data 2026-04-03 18.56.00_total.xlsx')

arial_path = r"C:\Windows\Fonts\arial.ttf"
if os.path.exists(arial_path):
    pdfmetrics.registerFont(TTFont('ArialUni', arial_path))
    FONT_NAME = 'ArialUni'
else:
    FONT_NAME = 'Helvetica'
    print("WARNING: arial.ttf not found, Hungarian characters may not render correctly.")

def safe_filename(name):
    name = str(name).strip()
    return re.sub(r'[\\/:*?"<>|]+', '_', name) or "UNKNOWN"

def draw_header(c, y):
    c.setFont(FONT_NAME, 14)
    c.drawString(72, y, "Name")
    c.line(72, y - 4, 540, y - 4)
    c.setFont(FONT_NAME, 12)
    return y - 24

def export_list_to_pdf(data_list, pdf_filename):
    c = canvas.Canvas(pdf_filename, pagesize=letter)
    c.setFont(FONT_NAME, 12)

    x = 72
    y = draw_header(c, 740)
    max_lines_per_page = 32
    line_count = 0

    for item in data_list:
        c.drawString(x, y, str(item))
        y -= 20
        line_count += 1

        if line_count >= max_lines_per_page:
            c.showPage()
            c.setFont(FONT_NAME, 12)
            y = draw_header(c, 740)
            line_count = 0

    c.save()

os.makedirs(tempfolder, exist_ok=True)

for list_code, group in df_pdf.groupby('ListCode', dropna=False):
    code = safe_filename(list_code)
    items = group['Name'].dropna().astype(str).tolist()
    if not items:
        continue
    pdf_path = os.path.join(tempfolder, f"HU CBH data 2026-04-03 18.56.00- {code}.pdf")
    export_list_to_pdf(items, pdf_path)
